# SIRAG Gemma 4 E4B — Self-Improving RAG (inference only)

**No fine-tuning.** Generator is [`google/gemma-4-E4B-it`](https://huggingface.co/google/gemma-4-E4B-it) loaded in **8-bit** via bitsandbytes (lighter/faster than 12B on T4).

Pipeline: crawl official [Mihon](https://mihon.app) docs → heading-aware chunks → hybrid FAISS + BM25 + RRF + cross-encoder rerank → Gemma HyDE / route / generate / gap re-retrieve → NLI (+ optional Gemma) faithfulness → SQLite memory with verified question→chunk aliases and distilled lessons.

### Self-improve mechanisms (weights frozen)
1. **SQLite memory** — reuse / evidence / blacklist from past verified & corrected answers
2. **HyDE + query rewrite** — Gemma drafts a short hypothetical answer used as an extra dense query
3. **CRAG-style pre-gate** — weak retrieval → rewrite once or abstain
4. **Sentence-level support check** — drop unsupported sentences or abstain
5. **Gap-targeted re-retrieve** — one extra retrieve on unsupported claims
6. **Verified aliases (ERM-lite)** — thumbs-up adds the question as a FAISS/BM25 key pointing at supporting chunks
7. **Lesson distillation** — on feedback, Gemma writes a one-line rule injected on later similar asks
8. **Adaptive route** — faq / multistep / ood heuristic (+ optional Gemma classify)

### Speed knobs
Set `USE_HYDE=False` and/or `USE_ROUTE=False` in the config cell to skip extra Gemma passes when you want faster answers.

### Gated model access
Accept the Gemma license on Hugging Face for `google/gemma-4-E4B-it`, then add a Kaggle Secret named **`HF_TOKEN`** (read token) before Run All.


## 1. Install dependencies

Needs a **recent** `transformers` that knows `gemma4_unified`. Restart the session once if the install upgrades an already-imported package.


In [1]:
%pip -q install -U "transformers>=4.57.0" "accelerate>=1.0.0" "bitsandbytes>=0.45.0" \
  "sentence-transformers>=3,<6" sentencepiece faiss-cpu rank-bm25 \
  trafilatura beautifulsoup4 pymupdf python-docx openpyxl scikit-learn tqdm huggingface_hub

import transformers
print("transformers", transformers.__version__)
print("SUCCESS: installs done.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
transformers 5.22.1
SUCCESS: installs done.


## 2. Imports and runtime


In [2]:
import gc
import hashlib
import json
import logging
import os
import random
import re
import shutil
import sqlite3
import time
import warnings
from collections import Counter, deque
from contextlib import closing
from pathlib import Path
from urllib.parse import urldefrag, urljoin, urlparse

import faiss
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import requests
import torch
import trafilatura
import transformers
from bs4 import BeautifulSoup
from docx import Document
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=FutureWarning)
# bitsandbytes MatMul8bitLt spam is logging.warning (NOT warnings.warn) — filter both.
warnings.filterwarnings("ignore", message=r".*MatMul8bitLt.*")
warnings.filterwarnings("ignore", message=r".*inputs will be cast.*float16.*")
warnings.filterwarnings("ignore", message=r".*cast from.*to float16.*")


def _silence_bnb_cast_spam():
    """Stop continuous MatMul8bitLt cast logs from bitsandbytes (fires every forward)."""
    for _name in (
        "bitsandbytes",
        "bitsandbytes.autograd",
        "bitsandbytes.autograd._functions",
        "bitsandbytes.nn",
        "bitsandbytes.nn.modules",
    ):
        logging.getLogger(_name).setLevel(logging.ERROR)
    # Belt-and-suspenders if a handler was already attached at WARNING.
    for _name in ("bitsandbytes", "bitsandbytes.autograd._functions"):
        _log = logging.getLogger(_name)
        for _h in list(_log.handlers):
            if getattr(_h, "level", 0) and _h.level < logging.ERROR:
                _h.setLevel(logging.ERROR)


_silence_bnb_cast_spam()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("SUCCESS: imports ready.")



/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


device: cuda
GPU: Tesla T4 | VRAM GB: 15.64
SUCCESS: imports ready.


## 3. Configuration


In [3]:
PROJECT_NAME = "SIRAG"
# Prefer Kaggle working dir; fall back to Colab Drive if present; else local.
if Path("/kaggle/working").exists():
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
elif Path("/content").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    except Exception:
        PROJECT_DIR = Path(".") / PROJECT_NAME
else:
    PROJECT_DIR = Path(".") / PROJECT_NAME
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_DIR.resolve())

# ---- Models (inference only; no Trainer / no fine-tune) ----
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL_NAME = "google/gemma-4-E4B-it"
NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
USE_RERANKER = True
LOAD_IN_8BIT = True  # keep 8-bit for headroom with embedder + reranker + NLI on T4
ENABLE_THINKING = False  # keep answers short for RAG

# ---- Corpus ----
SITE_HOST = "mihon.app"
START_URLS = [
    "https://mihon.app/",
    "https://mihon.app/docs/guides/getting-started",
    "https://mihon.app/docs/faq/general",
]
MAX_PAGES = 80
MAX_DEPTH = 3
USE_CORPUS_CACHE = True
CORPUS_CACHE = PROJECT_DIR / "mihon_corpus.json"
FILE_PATHS = []
URLS = []
CHUNK_SIZE = 600
CHUNK_OVERLAP = 100

# ---- Retrieval / generation ----
TOP_K = 5
RERANK_POOL = 30
MAX_NEW_TOKENS = 512
CRAG_RERANK_MIN = -2.5          # below this max rerank score → rewrite or abstain (ms-marco logits)
CRAG_NLI_MIN = 0.35             # optional chunk-vs-question entailment floor
GAP_RETRIEVE = True
# Speed: set either False to skip an extra Gemma forward pass per ask()
USE_HYDE = True                 # False → skip HyDE generate (faster)
USE_ROUTE = True                # False → heuristic route only, no Gemma classify (faster)
USE_GEMMA_ROUTE = USE_ROUTE     # alias kept for older cells
USE_GEMMA_LESSONS = True

# ---- Memory ----
USE_MEMORY = True
MEMORY_MIN_SIM = 0.78
MEMORY_REUSE_SIM = 0.92
MEMORY_TOP_K = 2
BLACKLIST_SIM = 0.85
BLACKLIST_F1 = 0.85
ACCEPT_SCORE = 0.99
MIN_SCORE = 0.55
ABSTAIN_ANSWER = "The documentation I have does not cover this reliably."

print("Generator:", GEN_MODEL_NAME, "| 8-bit:", LOAD_IN_8BIT, "| reranker:", USE_RERANKER)
print("USE_HYDE:", USE_HYDE, "| USE_ROUTE:", USE_ROUTE)
print("SUCCESS: config ready.")


Project directory: /kaggle/working/SIRAG
Generator: google/gemma-4-E4B-it | 8-bit: True | reranker: True
USE_HYDE: True | USE_ROUTE: True
SUCCESS: config ready.


## 4. Hugging Face token + load embedder / reranker

Add Kaggle Secret **`HF_TOKEN`** after accepting the Gemma license. Without it, model download will fail on the gated repo.


In [4]:
def _resolve_hf_token():
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if tok:
        return tok.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
            os.environ["HUGGINGFACE_HUB_TOKEN"] = tok
            return tok.strip()
    except Exception as exc:
        print("[HF_TOKEN] Kaggle Secrets not available:", type(exc).__name__, exc)
    return None

HF_TOKEN = _resolve_hf_token()
if HF_TOKEN:
    print("HF_TOKEN: present (len=%d)" % len(HF_TOKEN))
else:
    print("WARNING: HF_TOKEN missing. Accept https://huggingface.co/google/gemma-4-E4B-it and add Kaggle Secret HF_TOKEN.")

embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
reranker = CrossEncoder(RERANK_MODEL_NAME, device=DEVICE) if USE_RERANKER else None
print("embedder:", EMBED_MODEL_NAME)
print("reranker:", RERANK_MODEL_NAME if reranker else None)
print("SUCCESS: embedder/reranker loaded.")


[HF_TOKEN] Kaggle Secrets not available: ConnectionError Connection error trying to communicate with service.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (13 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

embedder: sentence-transformers/all-MiniLM-L6-v2
reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
SUCCESS: embedder/reranker loaded.


## 5. Load Gemma 4 E4B-it (8-bit)

Uses `AutoModelForMultimodalLM` + `AutoProcessor` as on the model card. Text-only chat path; thinking disabled for RAG. E4B is much lighter than 12B on a Kaggle T4.


In [5]:
from transformers import AutoProcessor, BitsAndBytesConfig

try:
    from transformers import AutoModelForMultimodalLM
except ImportError as exc:
    raise ImportError(
        "AutoModelForMultimodalLM missing — upgrade transformers "
        "(pip install -U transformers) and restart the kernel."
    ) from exc

processor = None
gen_model = None

def load_gemma():
    global processor, gen_model
    if gen_model is not None:
        return gen_model, processor
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN required for gated google/gemma-4-E4B-it")
    # Re-apply before first forward: bnb logs MatMul8bitLt cast via logging.warning every matmul.
    _silence_bnb_cast_spam()
    bnb = BitsAndBytesConfig(load_in_8bit=True) if LOAD_IN_8BIT else None
    print("Loading", GEN_MODEL_NAME, "8-bit=", LOAD_IN_8BIT, "...")
    processor = AutoProcessor.from_pretrained(GEN_MODEL_NAME, token=HF_TOKEN)
    # 8-bit path casts non-fp16 activations → spam; load fp16 so A.dtype already float16.
    compute_dtype = torch.float16 if LOAD_IN_8BIT else "auto"
    kwargs = dict(device_map="auto", token=HF_TOKEN, dtype=compute_dtype)
    if bnb is not None:
        kwargs["quantization_config"] = bnb
    try:
        gen_model = AutoModelForMultimodalLM.from_pretrained(GEN_MODEL_NAME, **kwargs)
    except Exception as first_exc:
        print("[WARN] AutoModelForMultimodalLM failed:", type(first_exc).__name__, first_exc)
        print("Falling back to AutoModelForCausalLM + AutoTokenizer ...")
        from transformers import AutoModelForCausalLM, AutoTokenizer
        processor = AutoTokenizer.from_pretrained(GEN_MODEL_NAME, token=HF_TOKEN)
        if getattr(processor, "pad_token", None) is None and getattr(processor, "eos_token", None):
            processor.pad_token = processor.eos_token
        gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_NAME, **kwargs)
    gen_model.eval()
    _silence_bnb_cast_spam()
    print("Gemma loaded. device_map:", getattr(gen_model, "hf_device_map", None), "| dtype:", compute_dtype)
    return gen_model, processor

# Load now so later cells can assume it exists (comment out to lazy-load).
load_gemma()
print("SUCCESS: Gemma ready.")



RuntimeError: HF_TOKEN required for gated google/gemma-4-E4B-it

## 6. Document ingestion helpers


In [ ]:
def _clean_text(text):
    text = (text or "").replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _html_to_text(html):
    return trafilatura.extract(html) or BeautifulSoup(html, "html.parser").get_text(" ", strip=True)


def _html_to_structured(html, source):
    """Prefer heading-aware sections from HTML; fall back to flat text."""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "noscript"]):
        tag.decompose()
    title = (soup.title.get_text(" ", strip=True) if soup.title else "") or source
    main = soup.find("main") or soup.find("article") or soup.body or soup
    sections = []
    path = []
    buf = []

    def flush():
        text = _clean_text("\n".join(buf))
        buf.clear()
        if text:
            heading = " > ".join(path) if path else title
            sections.append({"title": title, "heading": heading, "text": text, "source": source})

    for el in main.descendants:
        if getattr(el, "name", None) in {"h1", "h2", "h3", "h4"}:
            flush()
            level = int(el.name[1])
            path = path[: max(0, level - 1)]
            path.append(el.get_text(" ", strip=True))
        elif getattr(el, "name", None) in {"p", "li", "pre", "td", "th"}:
            t = el.get_text(" ", strip=True)
            if t:
                buf.append(t)
    flush()
    if sections:
        return sections
    flat = _clean_text(_html_to_text(html))
    return [{"title": title, "heading": title, "text": flat, "source": source}] if flat else []


def load_file(path):
    path = Path(path)
    ext = path.suffix.lower()
    if ext == ".pdf":
        with fitz.open(str(path)) as pdf:
            text = "\n".join(page.get_text("text") for page in pdf)
    elif ext in {".txt", ".md"}:
        text = path.read_text(encoding="utf-8", errors="ignore")
    elif ext in {".html", ".htm"}:
        return [{"source": str(path), "text": s["text"], "title": s["title"], "heading": s["heading"]}
                for s in _html_to_structured(path.read_text(encoding="utf-8", errors="ignore"), str(path))]
    elif ext == ".docx":
        doc = Document(str(path))
        text = "\n".join(p.text for p in doc.paragraphs)
    elif ext == ".csv":
        text = pd.read_csv(path).fillna("").to_csv(index=False)
    elif ext == ".xlsx":
        sheets = pd.read_excel(path, sheet_name=None)
        text = "\n\n".join(f"Sheet: {name}\n{df.fillna('').to_csv(index=False)}" for name, df in sheets.items())
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    text = _clean_text(text)
    if not text:
        raise ValueError(f"No extractable text in {path.name}")
    return [{"source": str(path), "text": text, "title": path.name, "heading": path.name}]


def load_url(url, timeout=25, max_chars=1_500_000):
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("Provide a valid public http(s) URL.")
    response = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0 (compatible; SIRAG/1.0)"})
    response.raise_for_status()
    body = response.text[:max_chars]
    is_html = "html" in response.headers.get("content-type", "").lower() or "<html" in body[:1000].lower()
    if is_html:
        return _html_to_structured(body, url)
    text = _clean_text(body)
    return [{"source": url, "text": text, "title": url, "heading": url}] if text else []


def _canonical_source(source):
    parsed = urlparse(str(source).strip())
    if parsed.scheme and parsed.netloc:
        path = re.sub(r"/+", "/", parsed.path or "/")
        if path != "/":
            path = path.rstrip("/")
        return parsed.scheme.lower(), parsed.netloc.lower(), path, parsed.query
    return (str(Path(source).expanduser()),)


def load_sources(file_paths=None, urls=None):
    docs, seen = [], set()
    for path in file_paths or []:
        key = _canonical_source(path)
        if key in seen:
            continue
        seen.add(key)
        docs.extend(load_file(path))
    for url in urls or []:
        key = _canonical_source(url)
        if key in seen:
            continue
        seen.add(key)
        try:
            docs.extend(load_url(url))
        except Exception as exc:
            print(f"[INGESTION WARNING] Skipping {url}: {type(exc).__name__}: {exc}")
    return docs

print("SUCCESS: ingestion helpers ready.")


## 7. Recursive official-site crawl


In [ ]:
_SKIP_EXT = re.compile(r"\.(?:apk|zip|pdf|png|jpe?g|gif|svg|webp|ico|woff2?|ttf|css|js|json|xml|txt)$", re.I)


def _normalize_url(url):
    url, _ = urldefrag(url.strip())
    p = urlparse(url)
    path = re.sub(r"/+", "/", p.path or "/")
    if path != "/":
        path = path.rstrip("/")
    return p._replace(scheme=p.scheme.lower(), netloc=p.netloc.lower(), path=path, params="", query="").geturl()


def crawl_site(start_urls, allowed_host, max_pages=80, max_depth=3, delay_seconds=0.25, timeout=25):
    """Breadth-first crawl; returns structured section docs with title/heading."""
    allowed_host = allowed_host.lower()

    def in_scope(u):
        p = urlparse(u)
        return p.scheme == "https" and p.netloc == allowed_host and not _SKIP_EXT.search(p.path)

    queue = deque((_normalize_url(u), 0) for u in start_urls)
    queued = {u for u, _ in queue}
    visited, seen_final, docs = set(), set(), []
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; SIRAG/1.0)"})

    while queue and len(visited) < max_pages:
        url, depth = queue.popleft()
        if not in_scope(url):
            continue
        visited.add(url)
        try:
            response = session.get(url, timeout=timeout)
            response.raise_for_status()
            final = _normalize_url(response.url)
            visited.add(final)
            if urlparse(final).netloc != allowed_host or final in seen_final:
                continue
            if "text/html" not in response.headers.get("content-type", "").lower():
                continue
            seen_final.add(final)
            html = response.text[:1_500_000]
            sections = _html_to_structured(html, final)
            docs.extend(sections)
            if depth < max_depth:
                for a in BeautifulSoup(html, "html.parser").select("a[href]"):
                    href = a["href"].strip()
                    if not href or href.startswith(("mailto:", "javascript:", "tel:", "#")):
                        continue
                    nxt = _normalize_url(urljoin(response.url, href))
                    if in_scope(nxt) and nxt not in queued and nxt not in visited:
                        queued.add(nxt)
                        queue.append((nxt, depth + 1))
        except Exception as exc:
            print(f"[CRAWL WARNING] {url}: {type(exc).__name__}: {exc}")
        time.sleep(delay_seconds)

    print(f"Crawl finished: {len(docs)} sections from {len(seen_final)} pages / {len(visited)} visited URLs.")
    return docs

print("SUCCESS: crawl ready.")


## 8. Corpus assembly (cached)


In [ ]:
def load_corpus_docs(use_cache=USE_CORPUS_CACHE):
    if use_cache and CORPUS_CACHE.exists():
        docs = json.loads(CORPUS_CACHE.read_text(encoding="utf-8"))
        print(f"Loaded {len(docs)} cached sections from {CORPUS_CACHE}")
    else:
        docs = crawl_site(START_URLS, SITE_HOST, max_pages=MAX_PAGES, max_depth=MAX_DEPTH)
        if docs:
            CORPUS_CACHE.write_text(json.dumps(docs, ensure_ascii=False), encoding="utf-8")
            print(f"Cached {len(docs)} sections -> {CORPUS_CACHE}")
    if FILE_PATHS or URLS:
        docs = docs + load_sources(FILE_PATHS, URLS)
    if not docs:
        raise ValueError("Corpus is empty. Check network access and START_URLS.")
    return docs

print("SUCCESS: corpus loader ready.")


## 9. Heading-aware chunking (contextual prefixes)


In [ ]:
def _window_chunk(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, min_chars=40):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Require chunk_size > overlap >= 0.")
    pieces, n, start = [], len(text), 0
    while start < n:
        end = min(n, start + chunk_size)
        if end < n:
            cut = max(text.rfind(" ", start + chunk_size // 2, end), text.rfind("\n", start + chunk_size // 2, end))
            if cut > start:
                end = cut
        piece = text[start:end].strip()
        if piece and (len(piece) >= min_chars or not pieces):
            pieces.append((start, piece))
        if end >= n:
            break
        nxt = end - overlap
        if nxt <= start:
            nxt = end
        m = re.compile(r"\s").search(text, nxt, nxt + 40)
        start = m.end() if m else nxt
    return pieces


def chunk_documents(docs, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, min_chars=40):
    """Prefix each chunk with title/heading path for contextual retrieval."""
    chunks, seen_text = [], set()
    for doc in docs:
        title = doc.get("title") or doc.get("source") or ""
        heading = doc.get("heading") or title
        prefix = f"[{title}]"
        if heading and heading != title:
            prefix = f"[{title} | {heading}]"
        body = doc["text"]
        for start, piece in _window_chunk(body, chunk_size=chunk_size, overlap=overlap, min_chars=min_chars):
            prefixed = f"{prefix}\n{piece}".strip()
            norm = re.sub(r"\s+", " ", prefixed).casefold()
            if norm in seen_text:
                continue
            seen_text.add(norm)
            chunk_id = hashlib.sha1(f'{doc["source"]}:{start}:{prefixed}'.encode("utf-8")).hexdigest()[:16]
            chunks.append({
                "chunk_id": chunk_id,
                "source": doc["source"],
                "title": title,
                "heading": heading,
                "start_char": start,
                "text": prefixed,
                "raw_text": piece,
                "is_alias": False,
            })
    if not chunks:
        raise ValueError("No chunks were created.")
    return chunks

print("SUCCESS: chunking ready.")


## 10. Hybrid retrieval: FAISS + BM25 + RRF + rerank + verified aliases


In [ ]:
def _tokenize(text):
    return re.findall(r"\w+", text.lower())


def token_f1(prediction, reference):
    p, r = Counter(_tokenize(prediction)), Counter(_tokenize(reference))
    if not p or not r:
        return 0.0
    overlap = sum((p & r).values())
    if overlap == 0:
        return 0.0
    precision = overlap / sum(p.values())
    recall = overlap / sum(r.values())
    return 2 * precision * recall / (precision + recall)


class HybridRetriever:
    """Dense FAISS IP + BM25 fused with RRF; optional cross-encoder rerank.
    Verified question aliases are extra FAISS/BM25 keys that resolve to real chunks (ERM-lite).
    """

    def __init__(self, chunks, embedder, reranker=None, rrf_k=60, batch_size=64):
        if not chunks:
            raise ValueError("Retriever requires at least one chunk.")
        self.embedder = embedder
        self.reranker = reranker
        self.rrf_k = rrf_k
        self.base_chunks = [c for c in chunks if not c.get("is_alias")]
        self.alias_records = []  # {question, chunk_ids, text}
        self._rebuild(self.base_chunks, batch_size=batch_size)

    def _rebuild(self, chunks, batch_size=64):
        self.chunks = list(chunks)
        # append alias pseudo-chunks at the end
        for rec in self.alias_records:
            self.chunks.append({
                "chunk_id": rec["alias_id"],
                "source": "alias://" + rec["alias_id"],
                "text": rec["question"],
                "is_alias": True,
                "target_chunk_ids": rec["chunk_ids"],
            })
        self.texts = [c["text"] for c in self.chunks]
        self.bm25 = BM25Okapi([_tokenize(t) for t in self.texts])
        vectors = self.embedder.encode(
            self.texts, batch_size=batch_size, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=True,
        ).astype("float32")
        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)
        self.id_to_idx = {c["chunk_id"]: i for i, c in enumerate(self.chunks) if not c.get("is_alias")}

    def add_question_alias(self, question, chunk_ids, batch_size=64):
        """ERM-lite: index the verified question string; retrieval expands to its chunk_ids."""
        question = (question or "").strip()
        if not question or not chunk_ids:
            return False
        alias_id = "alias:" + hashlib.sha1(question.encode("utf-8")).hexdigest()[:12]
        # dedupe
        self.alias_records = [r for r in self.alias_records if r["alias_id"] != alias_id]
        valid = [cid for cid in chunk_ids if cid in self.id_to_idx or any(c["chunk_id"] == cid for c in self.base_chunks)]
        if not valid:
            return False
        self.alias_records.append({"alias_id": alias_id, "question": question, "chunk_ids": valid})
        self._rebuild(self.base_chunks, batch_size=batch_size)
        print(f"Alias added: {alias_id!r} -> {valid}")
        return True

    def _expand(self, idx):
        c = self.chunks[idx]
        if c.get("is_alias"):
            out = []
            for cid in c.get("target_chunk_ids", []):
                j = self.id_to_idx.get(cid)
                if j is not None:
                    out.append(j)
            return out
        return [idx]

    def retrieve(self, query, top_k=TOP_K, dense_k=None, bm25_k=None, extra_dense_queries=None):
        n = len(self.chunks)
        top_k = max(1, min(int(top_k), max(1, len(self.base_chunks))))
        dense_k = min(n, dense_k or max(RERANK_POOL, top_k * 5))
        bm25_k = min(n, bm25_k or max(RERANK_POOL, top_k * 5))

        queries = [query] + list(extra_dense_queries or [])
        rankings = []
        for q in queries:
            qvec = self.embedder.encode([q], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
            _, dense_ids = self.index.search(qvec, dense_k)
            expanded = []
            for i in dense_ids[0]:
                if i < 0:
                    continue
                expanded.extend(self._expand(int(i)))
            # unique preserve order
            seen, order = set(), []
            for i in expanded:
                if i not in seen:
                    seen.add(i)
                    order.append(i)
            rankings.append(order)

        q_tokens = _tokenize(query)
        if q_tokens:
            scores = self.bm25.get_scores(q_tokens)
            order = np.argsort(scores)[::-1][:bm25_k]
            expanded = []
            for i in order:
                if scores[i] > 0:
                    expanded.extend(self._expand(int(i)))
            seen, bm_order = set(), []
            for i in expanded:
                if i not in seen:
                    seen.add(i)
                    bm_order.append(i)
            rankings.append(bm_order)

        fused = {}
        for ranking in rankings:
            for rank, idx in enumerate(ranking, start=1):
                fused[idx] = fused.get(idx, 0.0) + 1.0 / (self.rrf_k + rank)

        candidates, seen_text = [], set()
        for idx in sorted(fused, key=fused.get, reverse=True)[: max(RERANK_POOL, top_k * 4)]:
            if self.chunks[idx].get("is_alias"):
                continue
            norm = re.sub(r"\s+", " ", self.texts[idx]).strip().casefold()
            if norm in seen_text:
                continue
            seen_text.add(norm)
            candidates.append(dict(self.chunks[idx], retrieval_score=float(fused[idx])))

        if self.reranker is not None and candidates:
            scores = self.reranker.predict([(query, c["text"]) for c in candidates])
            for c, s in zip(candidates, scores):
                c["rerank_score"] = float(s)
            candidates.sort(key=lambda c: c["rerank_score"], reverse=True)
        return candidates[:top_k]

print("SUCCESS: HybridRetriever ready.")


## 11. Persistent SQLite logging + lessons


In [ ]:
LOCAL_DB = Path("/kaggle/working/SIRAG.sqlite3") if Path("/kaggle/working").exists() else (
    Path("/content/SIRAG.sqlite3") if Path("/content").exists() else Path("SIRAG.sqlite3")
)
DRIVE_DB = PROJECT_DIR / "SIRAG_gemma4.sqlite3"
if DRIVE_DB.exists() and not LOCAL_DB.exists():
    shutil.copy2(DRIVE_DB, LOCAL_DB)
DB_PATH = str(LOCAL_DB)


def sync_db_to_drive(path=DB_PATH):
    if str(path) != DB_PATH:
        return
    try:
        shutil.copy2(LOCAL_DB, DRIVE_DB)
    except OSError as exc:
        print(f"[DB SYNC WARNING] {exc}")


def _connect(path=DB_PATH):
    return closing(sqlite3.connect(path))


def _ensure_column(conn, table, column, ddl):
    if column not in {row[1] for row in conn.execute(f"PRAGMA table_info({table})")}:
        conn.execute(f"ALTER TABLE {table} ADD COLUMN {column} {ddl}")


def init_db(path=DB_PATH):
    with _connect(path) as conn:
        with conn:
            conn.execute("""CREATE TABLE IF NOT EXISTS runs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP,
                question TEXT NOT NULL,
                answer TEXT NOT NULL,
                iterations_json TEXT,
                evaluation_json TEXT
            )""")
            conn.execute("""CREATE TABLE IF NOT EXISTS lessons (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP,
                run_id INTEGER,
                question TEXT,
                lesson TEXT NOT NULL,
                embedding BLOB
            )""")
            for column, ddl in (
                ("retrieved_json", "TEXT"), ("embedding", "BLOB"), ("faithfulness", "REAL"),
                ("auto_status", "TEXT"), ("source", "TEXT"), ("feedback", "TEXT"),
                ("correction", "TEXT"), ("route", "TEXT"),
            ):
                _ensure_column(conn, "runs", column, ddl)


init_db()


def _auto_status_from_faith(faith):
    if (faith or {}).get("status") != "success":
        return "neutral"
    score = faith.get("faithfulness_score")
    if score == 1.0:
        return "verified"
    if score == 0.0 or faith.get("contradicted_claims", 0) > 0:
        return "bad"
    return "neutral"


def log_result(result, qvec=None, path=DB_PATH):
    faith = result.get("faithfulness") or {}
    source = result.get("source", "loop")
    auto = "neutral" if source == "abstain" else "verified" if source == "memory" else _auto_status_from_faith(faith)
    iterations = [{"iteration": h.get("iteration"), "answer": h.get("answer"), "note": h.get("critique") or h.get("note")}
                  for h in result.get("history", [])]
    evidence = [{"source": e.get("source"), "chunk_id": e.get("chunk_id")} for e in result.get("evidence", [])]
    evaluation = {
        "faithfulness": faith,
        "memory_used": result.get("memory"),
        "route": result.get("route"),
        "crag": result.get("crag"),
        "candidates": result.get("candidates"),
    }
    blob = None if qvec is None else np.asarray(qvec, dtype="float32").tobytes()
    with _connect(path) as conn:
        with conn:
            cur = conn.execute(
                "INSERT INTO runs(question, answer, iterations_json, evaluation_json, retrieved_json, embedding, "
                "faithfulness, auto_status, source, route) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                (result["question"], result["answer"], json.dumps(iterations), json.dumps(evaluation, default=str),
                 json.dumps(evidence), blob, faith.get("faithfulness_score"), auto, source, result.get("route")),
            )
            run_id = cur.lastrowid
    sync_db_to_drive(path)
    return run_id


def recent_runs(limit=20, path=DB_PATH):
    with _connect(path) as conn:
        return pd.read_sql_query(
            "SELECT id, created_at, question, answer, source, route, faithfulness, auto_status, feedback, correction "
            "FROM runs ORDER BY id DESC LIMIT ?",
            conn, params=(int(limit),),
        )


def store_lesson(run_id, question, lesson, qvec=None, path=DB_PATH):
    lesson = (lesson or "").strip()
    if not lesson:
        return None
    blob = None if qvec is None else np.asarray(qvec, dtype="float32").tobytes()
    with _connect(path) as conn:
        with conn:
            cur = conn.execute(
                "INSERT INTO lessons(run_id, question, lesson, embedding) VALUES (?, ?, ?, ?)",
                (run_id, question, lesson, blob),
            )
            lid = cur.lastrowid
    sync_db_to_drive(path)
    return lid


def recall_lessons(question, qvec=None, top_k=3, min_sim=0.72, path=DB_PATH):
    qvec = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype("float32")[0] if qvec is None else qvec
    with _connect(path) as conn:
        rows = conn.execute("SELECT id, question, lesson, embedding FROM lessons WHERE embedding IS NOT NULL").fetchall()
    out = []
    for rid, q, lesson, blob in rows:
        if len(blob) != qvec.nbytes:
            continue
        sim = float(np.frombuffer(blob, dtype="float32") @ qvec)
        if sim >= min_sim:
            out.append({"id": rid, "question": q, "lesson": lesson, "sim": sim})
    out.sort(key=lambda x: -x["sim"])
    return out[:top_k]

print("DB:", DB_PATH)
print("SUCCESS: sqlite ready.")


## 12. NLI faithfulness (cheap vote)


In [ ]:
NLI_MAX_LENGTH = 512
FAITHFULNESS_ENTAILMENT_THRESHOLD = 0.60
FAITHFULNESS_CONTRADICTION_THRESHOLD = 0.55
_nli = {"tokenizer": None, "model": None, "labels": None}


def _load_nli_model():
    if _nli["model"] is not None:
        return
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
    nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(DEVICE).eval()
    id2label = {int(k): re.sub(r"[^a-z]", "", str(v).lower()) for k, v in nli_model.config.id2label.items()}
    ids = {}
    for idx, label in id2label.items():
        for name, stem in (("entailment", "entail"), ("contradiction", "contrad"), ("neutral", "neutral")):
            if stem in label:
                ids[name] = idx
    if set(ids) != {"entailment", "contradiction", "neutral"}:
        if nli_model.config.num_labels == 3:
            ids = {"contradiction": 0, "entailment": 1, "neutral": 2}
        else:
            raise RuntimeError(f"Cannot map NLI labels safely: {id2label}")
    _nli.update(tokenizer=nli_tok, model=nli_model, labels=ids)


def split_into_claims(answer):
    text = re.sub(r"\[S\d+\]", "", str(answer or ""))
    text = re.sub(r"(?m)^\s*(?:[-*•]|\d+[.)])\s+", "", text)
    claims = []
    for piece in re.split(r"(?<=[.!?])\s+|\n+", text):
        piece = re.sub(r"\s+", " ", piece).strip(" \t-•")
        if not piece:
            continue
        if re.fullmatch(r"(?i)(i (do not|don't) know|the (provided )?context (does not|doesn't) say|insufficient evidence|the documentation i have does not cover this reliably)[.!]?", piece):
            continue
        if len(re.findall(r"\w+", piece)) >= 3:
            claims.append(piece)
    return claims


def _nli_probabilities(premises, hypotheses, batch_size=16):
    _load_nli_model()
    model, nli_tok, labels = _nli["model"], _nli["tokenizer"], _nli["labels"]
    device = next(model.parameters()).device
    results = []
    for start in range(0, len(premises), batch_size):
        enc = nli_tok(
            premises[start:start + batch_size], hypotheses[start:start + batch_size],
            padding=True, truncation=True, max_length=NLI_MAX_LENGTH, return_tensors="pt",
        ).to(device)
        with torch.inference_mode():
            probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
        for row in probs:
            results.append({name: float(row[idx]) for name, idx in labels.items()})
    return results


def evaluate_faithfulness(answer, retrieved, entailment_threshold=None, contradiction_threshold=None, batch_size=16):
    ent_thr = entailment_threshold or FAITHFULNESS_ENTAILMENT_THRESHOLD
    con_thr = contradiction_threshold or FAITHFULNESS_CONTRADICTION_THRESHOLD
    claims = split_into_claims(answer)
    evidence = [x for x in (retrieved or []) if str(x.get("text", "")).strip()]
    if not claims:
        return {"status": "not_applicable", "faithfulness_score": None, "supported_claims": 0,
                "unsupported_claims": 0, "contradicted_claims": 0, "claim_count": 0, "claims": [],
                "note": "No factual claims detected."}
    if not evidence:
        return {"status": "unavailable", "faithfulness_score": None, "supported_claims": 0,
                "unsupported_claims": len(claims), "contradicted_claims": 0, "claim_count": len(claims),
                "claims": [{"claim": c, "verdict": "unverifiable", "evidence_source": None} for c in claims],
                "note": "No evidence available."}

    premises, hypotheses, pair_map = [], [], []
    for ci, claim in enumerate(claims):
        for ei, item in enumerate(evidence):
            premises.append(item["text"])
            hypotheses.append(claim)
            pair_map.append((ci, ei))
    probs = _nli_probabilities(premises, hypotheses, batch_size=batch_size)
    per_claim = [[] for _ in claims]
    for (ci, ei), prob in zip(pair_map, probs):
        per_claim[ci].append((ei, prob))

    details = []
    for claim, scored in zip(claims, per_claim):
        best_ent = max(scored, key=lambda x: x[1]["entailment"])
        best_con = max(scored, key=lambda x: x[1]["contradiction"])
        ent_p, con_p = best_ent[1]["entailment"], best_con[1]["contradiction"]
        if ent_p >= ent_thr and ent_p > con_p:
            verdict, chosen = "supported", best_ent
        elif con_p >= con_thr and con_p > ent_p:
            verdict, chosen = "contradicted", best_con
        else:
            verdict, chosen = "not_supported_by_retrieved_evidence", best_ent
        item = evidence[chosen[0]]
        details.append({
            "claim": claim, "verdict": verdict,
            "entailment_probability": round(ent_p, 4),
            "contradiction_probability": round(con_p, 4),
            "evidence_source": item.get("source"), "evidence_chunk_id": item.get("chunk_id"),
        })

    supported = sum(d["verdict"] == "supported" for d in details)
    contradicted = sum(d["verdict"] == "contradicted" for d in details)
    return {
        "status": "success",
        "faithfulness_score": round(supported / len(details), 4),
        "supported_claims": supported,
        "unsupported_claims": len(details) - supported,
        "contradicted_claims": contradicted,
        "claim_count": len(details),
        "claims": details,
        "note": "NLI estimate of support by evidence, not ground-truth accuracy.",
    }


def drop_unsupported_sentences(answer, faith):
    """Keep only supported / non-evaluated sentences; drop contradicted & unsupported."""
    if not faith or faith.get("status") != "success":
        return answer, []
    bad = {d["claim"] for d in faith.get("claims", []) if d["verdict"] != "supported"}
    if not bad:
        return answer, []
    kept, dropped = [], []
    for sent in split_into_claims(answer) or [answer]:
        if sent in bad:
            dropped.append(sent)
        else:
            kept.append(sent)
    # Also keep non-claim glue if any
    if not kept:
        return "", dropped
    return " ".join(kept), dropped

print("SUCCESS: NLI faithfulness ready.")


## 13. Gemma chat helpers (text-only)


In [ ]:
def _norm(text):
    return re.sub(r"\s+", " ", (text or "").strip()).casefold()


def _messages_to_inputs(messages, enable_thinking=ENABLE_THINKING):
    """Build model inputs via processor chat template (multimodal or tokenizer fallback)."""
    model, proc = load_gemma()
    # Normalize content to plain strings for text-only
    norm_msgs = []
    for m in messages:
        content = m["content"]
        if isinstance(content, list):
            content = " ".join(
                (p.get("text") if isinstance(p, dict) else str(p)) for p in content
            )
        norm_msgs.append({"role": m["role"], "content": content})

    if hasattr(proc, "apply_chat_template"):
        try:
            inputs = proc.apply_chat_template(
                norm_msgs,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
            )
        except TypeError:
            # tokenizer path may not accept enable_thinking / return_dict the same way
            prompt = proc.apply_chat_template(
                norm_msgs, tokenize=False, add_generation_prompt=True,
            )
            inputs = proc(prompt, return_tensors="pt")
    else:
        raise RuntimeError("Processor/tokenizer lacks apply_chat_template")

    # Move tensors to model device
    device = getattr(model, "device", None)
    if device is None:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if hasattr(inputs, "items"):
        inputs = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in inputs.items()}
    else:
        inputs = inputs.to(device)
    return inputs


def _decode_new_tokens(inputs, outputs):
    model, proc = load_gemma()
    if isinstance(inputs, dict):
        input_len = inputs["input_ids"].shape[-1]
        seq = outputs[0][input_len:]
    else:
        input_len = inputs["input_ids"].shape[-1] if "input_ids" in inputs else 0
        seq = outputs[0][input_len:]
    if hasattr(proc, "decode"):
        text = proc.decode(seq, skip_special_tokens=False)
    else:
        text = proc.batch_decode(seq.unsqueeze(0), skip_special_tokens=False)[0]
    # Prefer parse_response when available
    if hasattr(proc, "parse_response"):
        try:
            parsed = proc.parse_response(text, prefix=inputs["input_ids"] if isinstance(inputs, dict) else None)
            if isinstance(parsed, dict):
                return (parsed.get("content") or parsed.get("answer") or parsed.get("text") or text).strip()
            if isinstance(parsed, str) and parsed.strip():
                return parsed.strip()
        except Exception:
            pass
    # Strip common channel / think tags
    text = re.sub(r"<\|channel\|>thought\n.*?<channel\|>", "", text, flags=re.S)
    text = re.sub(r"<\|.*?\|?>", "", text)
    return text.strip()


@torch.inference_mode()
def gemma_chat(system, user, max_new_tokens=MAX_NEW_TOKENS, temperature=0.2, top_p=0.9):
    """Text-only chat completion with Gemma 4 instruct."""
    model, _ = load_gemma()
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user})
    inputs = _messages_to_inputs(messages, enable_thinking=False)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=temperature > 0)
    if temperature > 0:
        gen_kwargs.update(temperature=temperature, top_p=top_p, top_k=64)
    outputs = model.generate(**inputs, **gen_kwargs) if isinstance(inputs, dict) else model.generate(inputs, **gen_kwargs)
    return _decode_new_tokens(inputs, outputs)


def gemma_hyde(question):
    sys = (
        "You write a short hypothetical documentation paragraph that would answer the user question. "
        "Do not say you are hypothetical. 2-4 sentences, Mihon Android reader docs style. No preamble."
    )
    return gemma_chat(sys, question, max_new_tokens=180, temperature=0.3)


def gemma_rewrite_query(question):
    sys = (
        "Rewrite the user question as a concise search query for Mihon documentation. "
        "Expand abbreviations. Output ONLY the query, no quotes."
    )
    return gemma_chat(sys, question, max_new_tokens=64, temperature=0.1)


# In-domain cues for Mihon / Tachiyomi-fork Android reader docs (mihon.app).
DOMAIN_KEYWORDS = (
    "mihon", "tachiyomi", "android", "reader", "extension", "extensions",
    "manga", "comic", "comics", "apk", "install", "backup", "restore",
    "source", "sources", "catalogue", "catalog", "library", "tracker",
    "download", "chapter", "chapters", "fork", "mihon.app", "docs",
    "documentation", "this site", "this app", "the site", "the app",
    "the docs", "official site", "website",
)


def _has_domain_keywords(question):
    q = (question or "").lower()
    return any(k in q for k in DOMAIN_KEYWORDS)


def gemma_route(question):
    sys = (
        "You classify questions for a RAG system whose ONLY corpus is Mihon "
        "(Tachiyomi-fork) Android manga/comic reader documentation at mihon.app.\n"
        "Return exactly one label: faq | multistep | ood\n"
        "faq = single-fact / yes-no / definition about Mihon or these docs\n"
        "multistep = how-to with several steps (install, setup, backup, extensions, etc.)\n"
        "ood = ONLY clearly unrelated topics (weather, stocks, cooking, politics, sports, etc.)\n"
        "Rules:\n"
        "- Questions mentioning Mihon, Tachiyomi, Android reader, extensions, manga/comic reader, "
        "install, sources, backup, catalogue, APK, or similar are NEVER ood.\n"
        "- Questions about 'this site', 'this app', 'the docs', or 'the website' refer to Mihon "
        "and are in-domain (usually faq).\n"
        "- When unsure between faq/multistep/ood, prefer faq or multistep — never ood.\n"
        "Reply with only the label."
    )
    label = gemma_chat(sys, question, max_new_tokens=8, temperature=0.0).strip().lower()
    for key in ("faq", "multistep", "ood"):
        if key in label:
            return key
    return "faq"


def heuristic_route(question):
    q = question.lower()
    ood_hints = ("weather", "stock", "recipe", "bitcoin", "who is the president", "cook", "politics")
    # True ood only when unrelated AND no Mihon-domain cues
    if any(h in q for h in ood_hints) and not _has_domain_keywords(question):
        return "ood"
    if re.search(r"\b(how (do|to)|steps?|install|setup|configure|migrate|backup|restore)\b", q):
        return "multistep"
    return "faq"


def resolve_route(question, gemma_label=None):
    """Combine Gemma route with heuristic + domain-keyword safety override.

    Abstain as ood ONLY when Gemma says ood AND heuristic says ood AND no domain keywords.
    Otherwise discard false Gemma ood and fall back to heuristic (or faq if keywords present).
    """
    heur = heuristic_route(question)
    in_domain = _has_domain_keywords(question)
    if gemma_label is None:
        return heur
    if gemma_label != "ood":
        return gemma_label
    # Gemma said ood — override if domain cues or heuristic disagrees
    if in_domain or heur != "ood":
        override = heur if heur != "ood" else "faq"
        print(f"[ROUTE OVERRIDE] gemma=ood -> {override} (heur={heur}, domain_kw={in_domain})")
        return override
    return "ood"


def gemma_distill_lesson(question, answer, feedback, correction=None):
    sys = (
        "Write ONE short rule (max 20 words) the assistant should remember for similar Mihon questions. "
        "No preamble. Imperative voice."
    )
    user = f"Question: {question}\nAnswer: {answer}\nFeedback: {feedback}\nCorrection: {correction or ''}"
    return gemma_chat(sys, user, max_new_tokens=40, temperature=0.2)

print("SUCCESS: Gemma helpers ready.")


## 14. Ask pipeline (self-improving, no fine-tune)


In [ ]:
def _embed_question(question):
    return embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype("float32")[0]


def _classify(row):
    out = []
    fb = row["feedback"]
    if row["correction"]:
        out.append(("gold", row["correction"]))
    if fb == "bad" or (fb != "good" and row["auto_status"] == "bad"):
        out.append(("bad", row["answer"]))
    elif fb == "good" or (not row["correction"] and row["auto_status"] == "verified"):
        out.append(("verified", row["answer"]))
    return out


def _memory_chunk(m):
    label = "Corrected answer" if m["status"] == "gold" else "Verified answer"
    return {
        "chunk_id": f"mem:{m['run_id']}:{m['status']}",
        "source": f"memory://run/{m['run_id']}",
        "text": f'{label} to a similar question ("{m["question"]}"): {m["answer"]}',
        "memory": True,
    }


def recall_memory(question, qvec=None, path=DB_PATH):
    memory = {"evidence": [], "bad_answers": [], "reuse": None, "matches": []}
    qvec = _embed_question(question) if qvec is None else qvec
    with _connect(path) as conn:
        rows = conn.execute(
            "SELECT id, question, answer, auto_status, feedback, correction, embedding "
            "FROM runs WHERE embedding IS NOT NULL"
        ).fetchall()
    rows = [r for r in rows if len(r[6]) == qvec.nbytes]
    if not rows:
        return memory
    sims = np.vstack([np.frombuffer(r[6], dtype="float32") for r in rows]) @ qvec
    floor = min(MEMORY_MIN_SIM, BLACKLIST_SIM)
    good = []
    for i in np.argsort(-sims):
        sim = float(sims[i])
        if sim < floor:
            break
        rid, q, ans, auto, fb, corr, _ = rows[i]
        for status, text in _classify({"answer": ans, "auto_status": auto, "feedback": fb, "correction": corr}):
            if status == "bad":
                if sim >= BLACKLIST_SIM:
                    memory["bad_answers"].append(text)
            elif sim >= MEMORY_MIN_SIM:
                good.append({"run_id": rid, "question": q, "answer": text, "status": status, "sim": sim})
    banned = {_norm(b) for b in memory["bad_answers"]}
    seen, kept = set(), []
    for g in sorted(good, key=lambda g: (g["status"] != "gold", -g["sim"])):
        key = _norm(g["answer"])
        if key in banned or key in seen:
            continue
        seen.add(key)
        kept.append(g)
    memory["evidence"] = [_memory_chunk(g) for g in kept[:MEMORY_TOP_K]]
    memory["matches"] = [{k: g[k] for k in ("run_id", "question", "status", "sim")} for g in kept[:MEMORY_TOP_K]]
    memory["reuse"] = next((g for g in kept if g["sim"] >= MEMORY_REUSE_SIM), None)
    return memory


def _format_evidence(evidence):
    blocks = []
    for i, e in enumerate(evidence, 1):
        blocks.append(f"[S{i}] ({e.get('source','')})\n{e.get('text','')}")
    return "\n\n".join(blocks)


def generate_grounded(question, evidence, lessons=None):
    lesson_txt = ""
    if lessons:
        lesson_txt = "Lessons from prior feedback:\n- " + "\n- ".join(l["lesson"] for l in lessons) + "\n\n"
    sys = (
        "You are SIRAG, answering ONLY from the provided Mihon documentation excerpts. "
        "Cite sources inline as [S1], [S2], ... Do not invent APIs or settings. "
        "If the excerpts are insufficient, say so briefly."
    )
    user = f"{lesson_txt}Question: {question}\n\nExcerpts:\n{_format_evidence(evidence)}\n\nAnswer:"
    return gemma_chat(sys, user, max_new_tokens=MAX_NEW_TOKENS, temperature=0.2)


def crag_gate(question, evidence):
    """Return status: good | rewrite | abstain, plus diagnostics."""
    if not evidence:
        return {"status": "abstain", "reason": "no_evidence", "max_rerank": None}
    scores = [c.get("rerank_score") for c in evidence if c.get("rerank_score") is not None]
    max_rr = max(scores) if scores else None
    # Cheap question-vs-top-chunk NLI entailment as soft signal
    top = evidence[0]["text"]
    try:
        probs = _nli_probabilities([top], [question], batch_size=1)[0]
        q_ent = probs["entailment"]
    except Exception:
        q_ent = None
    if max_rr is not None and max_rr < CRAG_RERANK_MIN and (q_ent is None or q_ent < CRAG_NLI_MIN):
        return {"status": "abstain", "reason": "weak_retrieval", "max_rerank": max_rr, "q_entail": q_ent}
    if max_rr is not None and max_rr < 0.0:
        return {"status": "rewrite", "reason": "marginal_retrieval", "max_rerank": max_rr, "q_entail": q_ent}
    return {"status": "good", "reason": "ok", "max_rerank": max_rr, "q_entail": q_ent}


def ask(question, retriever=None, top_k=TOP_K, use_memory=None, log=True):
    """Inference-only self-improving ask()."""
    use_memory = USE_MEMORY if use_memory is None else use_memory
    if retriever is None:
        retriever = globals().get("retriever")
    if retriever is None:
        raise RuntimeError("Build the index first.")
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question cannot be empty.")
    question = question.strip()
    history = []
    qvec = _embed_question(question)

    # Route (Gemma + in-domain safety override; never false-ood Mihon questions)
    if USE_ROUTE:
        gemma_label = gemma_route(question)
        route = resolve_route(question, gemma_label)
    else:
        route = heuristic_route(question)
    if route == "ood":
        result = {
            "question": question, "answer": ABSTAIN_ANSWER, "history": [{"iteration": 0, "answer": ABSTAIN_ANSWER, "note": "ood"}],
            "retrieved": [], "evidence": [], "faithfulness": {"status": "abstained", "faithfulness_score": None},
            "source": "abstain", "route": route, "memory": [], "crag": None, "candidates": [], "run_id": None,
        }
        if log:
            result["run_id"] = log_result(result, qvec)
        return result

    # Memory
    memory = recall_memory(question, qvec) if use_memory else {"evidence": [], "bad_answers": [], "reuse": None, "matches": []}
    lessons = recall_lessons(question, qvec) if USE_GEMMA_LESSONS else []
    if memory.get("reuse"):
        reuse = memory["reuse"]
        result = {
            "question": question, "answer": reuse["answer"],
            "history": [{"iteration": 0, "answer": reuse["answer"], "note": f"memory reuse {reuse['status']}"}],
            "retrieved": [_memory_chunk(reuse)], "evidence": [_memory_chunk(reuse)],
            "faithfulness": {"status": f"reused_{reuse['status']}", "faithfulness_score": 1.0},
            "source": "memory", "route": route, "memory": memory["matches"], "crag": None,
            "candidates": [], "lessons": lessons, "run_id": None,
        }
        if log:
            result["run_id"] = log_result(result, qvec)
        return result

    # HyDE + rewrite
    extra_q = []
    rewritten = None
    hyde_text = None
    if USE_HYDE:
        try:
            hyde_text = gemma_hyde(question)
            if hyde_text:
                extra_q.append(hyde_text)
                history.append({"iteration": 0, "answer": None, "note": "hyde", "hyde": hyde_text[:240]})
        except Exception as exc:
            print("[HYDE WARNING]", type(exc).__name__, exc)

    evidence = retriever.retrieve(question, top_k=top_k, extra_dense_queries=extra_q)
    evidence = memory["evidence"] + evidence
    crag = crag_gate(question, [e for e in evidence if not e.get("memory")])

    if crag["status"] == "rewrite":
        try:
            rewritten = gemma_rewrite_query(question)
            history.append({"iteration": 1, "answer": None, "note": "query_rewrite", "rewrite": rewritten})
            evidence = memory["evidence"] + retriever.retrieve(
                rewritten or question, top_k=top_k,
                extra_dense_queries=extra_q + ([rewritten] if rewritten else []),
            )
            crag = crag_gate(question, [e for e in evidence if not e.get("memory")])
        except Exception as exc:
            print("[REWRITE WARNING]", type(exc).__name__, exc)

    if crag["status"] == "abstain" and route != "multistep":
        result = {
            "question": question, "answer": ABSTAIN_ANSWER,
            "history": history + [{"iteration": len(history), "answer": ABSTAIN_ANSWER, "note": "crag_abstain"}],
            "retrieved": evidence, "evidence": evidence,
            "faithfulness": {"status": "abstained", "faithfulness_score": None},
            "source": "abstain", "route": route, "memory": memory["matches"], "crag": crag,
            "candidates": [], "lessons": lessons, "run_id": None,
        }
        if log:
            result["run_id"] = log_result(result, qvec)
        return result

    # Generate
    answer = generate_grounded(question, evidence[:top_k], lessons=lessons)
    history.append({"iteration": len(history), "answer": answer, "note": "generate"})

    # Faithfulness + drop unsupported
    faith = evaluate_faithfulness(answer, evidence[:top_k])
    cleaned, dropped = drop_unsupported_sentences(answer, faith)
    if dropped and cleaned:
        answer = cleaned
        history.append({"iteration": len(history), "answer": answer, "note": f"dropped_{len(dropped)}_unsupported"})
        faith = evaluate_faithfulness(answer, evidence[:top_k])

    # Gap re-retrieve once
    if GAP_RETRIEVE and faith.get("status") == "success" and faith.get("unsupported_claims", 0) > 0:
        gaps = [d["claim"] for d in faith.get("claims", []) if d["verdict"] != "supported"][:3]
        if gaps:
            gap_q = " ".join(gaps)
            more = retriever.retrieve(gap_q, top_k=top_k)
            # merge unique
            seen = {e.get("chunk_id") for e in evidence}
            for m in more:
                if m.get("chunk_id") not in seen:
                    evidence.append(m)
                    seen.add(m.get("chunk_id"))
            answer = generate_grounded(question, evidence[:top_k + 2], lessons=lessons)
            history.append({"iteration": len(history), "answer": answer, "note": "gap_rereceive"})
            faith = evaluate_faithfulness(answer, evidence[:top_k + 2])
            cleaned, dropped = drop_unsupported_sentences(answer, faith)
            if dropped and cleaned:
                answer = cleaned
                faith = evaluate_faithfulness(answer, evidence[:top_k + 2])

    # Blacklist check
    if any(_norm(answer) == _norm(b) or token_f1(answer, b) >= BLACKLIST_F1 for b in memory.get("bad_answers", [])):
        answer, source = ABSTAIN_ANSWER, "abstain"
        faith = {"status": "abstained", "faithfulness_score": None, "note": "matched_blacklist"}
    else:
        score = faith.get("faithfulness_score")
        if score is not None and score < MIN_SCORE and faith.get("contradicted_claims", 0) > 0:
            answer, source = ABSTAIN_ANSWER, "abstain"
            faith = {**faith, "status": "abstained"}
        else:
            source = "gemma"

    result = {
        "question": question, "answer": answer, "history": history,
        "retrieved": evidence, "evidence": evidence[: top_k + 2],
        "faithfulness": faith, "source": source, "route": route,
        "memory": memory["matches"], "crag": crag, "lessons": lessons,
        "hyde": (hyde_text or "")[:300], "rewrite": rewritten,
        "candidates": [], "run_id": None,
    }
    if log:
        result["run_id"] = log_result(result, qvec)
    return result


def give_feedback(run_id, good=None, correction=None, path=DB_PATH, add_alias=True):
    """Rate a run. On good/correction: optional alias into FAISS + Gemma lesson."""
    if good is None and not (correction and correction.strip()):
        raise ValueError("Pass good=True/False and/or correction='...'.")
    feedback = "bad" if (correction and correction.strip()) else ("good" if good else "bad")
    corr = correction.strip() if correction and correction.strip() else None
    with _connect(path) as conn:
        row = conn.execute(
            "SELECT question, answer, retrieved_json, embedding FROM runs WHERE id = ?", (int(run_id),)
        ).fetchone()
        if not row:
            raise KeyError(f"No run with id {run_id}.")
        with conn:
            conn.execute(
                "UPDATE runs SET feedback = ?, correction = COALESCE(?, correction) WHERE id = ?",
                (feedback, corr, int(run_id)),
            )
    sync_db_to_drive(path)
    question, answer, retrieved_json, emb_blob = row
    print(f"Run {run_id} marked '{feedback}'" + (" with correction." if corr else "."))

    # Alias + lesson on positive signal
    if feedback == "good" or corr:
        try:
            evidence = json.loads(retrieved_json) if retrieved_json else []
            chunk_ids = [e.get("chunk_id") for e in evidence if e.get("chunk_id") and not str(e.get("chunk_id")).startswith("mem:")]
            if add_alias and chunk_ids and globals().get("retriever") is not None:
                retriever.add_question_alias(question, chunk_ids)
        except Exception as exc:
            print("[ALIAS WARNING]", type(exc).__name__, exc)
        if USE_GEMMA_LESSONS:
            try:
                qvec = np.frombuffer(emb_blob, dtype="float32") if emb_blob else _embed_question(question)
                lesson = gemma_distill_lesson(question, corr or answer, feedback, corr)
                lid = store_lesson(run_id, question, lesson, qvec=qvec)
                print(f"Lesson {lid}: {lesson}")
            except Exception as exc:
                print("[LESSON WARNING]", type(exc).__name__, exc)

print("SUCCESS: ask/give_feedback ready.")


## 15. Report helper


In [ ]:
def show_report(result, run_faithfulness=True):
    print(result.get("answer", ""))
    print("\n--- RAG report ---")
    if result.get("run_id") is not None:
        print("Run id:", result["run_id"], "| source:", result.get("source"), "| route:", result.get("route"))
    if result.get("crag"):
        print("CRAG:", result["crag"])
    if result.get("hyde"):
        print("HyDE:", result["hyde"][:200])
    if result.get("rewrite"):
        print("Rewrite:", result["rewrite"])
    for m in result.get("memory", []) or []:
        print(f"Memory: run {m['run_id']} ({m['status']}, sim {m['sim']:.2f}) - {m['question']}")
    for les in result.get("lessons", []) or []:
        print(f"Lesson (sim {les['sim']:.2f}): {les['lesson']}")
    faith = result.get("faithfulness")
    if faith is None and run_faithfulness:
        try:
            faith = evaluate_faithfulness(result.get("answer", ""), result.get("evidence", []))
            result["faithfulness"] = faith
        except Exception as exc:
            print("Faithfulness failed:", repr(exc))
            faith = None
    if faith:
        print("Faithfulness status:", faith.get("status"), "| score:", faith.get("faithfulness_score"))
        for claim in faith.get("claims", [])[:8]:
            print(f"- [{claim['verdict']}] {claim['claim']}")
    print("\n--- Evidence sources ---")
    for i, item in enumerate(result.get("evidence", []) or [], start=1):
        print(f"[S{i}] {item.get('source', 'Unknown')}")
    return result

print("SUCCESS: show_report ready.")


## 16. Smoke tests


In [ ]:
def run_smoke_tests():
    assert chunk_documents([{"source": "https://mihon.app/x", "text": "Hello world. " * 40, "title": "T", "heading": "H"}])
    toy = chunk_documents([{"source": "s", "text": "Mihon is an Android free and open source manga reader. " * 20, "title": "FAQ", "heading": "General"}])
    r = HybridRetriever(toy, embedder, reranker=reranker)
    hits = r.retrieve("What is Mihon?", top_k=2)
    assert hits and "text" in hits[0]
    r.add_question_alias("What is Mihon exactly?", [hits[0]["chunk_id"]])
    hits2 = r.retrieve("What is Mihon exactly?", top_k=2)
    assert hits2
    # tiny faithfulness
    faith = evaluate_faithfulness(hits[0]["text"][:120], hits[:1])
    assert faith["status"] in {"success", "not_applicable", "unavailable"}
    print("SMOKE OK: chunk=%d hits=%d alias_hits=%d faith=%s" % (len(toy), len(hits), len(hits2), faith["status"]))

run_smoke_tests()


## 17. Build the corpus index

Crawls (or loads cache), chunks with heading prefixes, builds hybrid retriever. **No training.**


In [ ]:
docs = load_corpus_docs()
chunks = chunk_documents(docs)
print(f"docs/sections={len(docs)} chunks={len(chunks)}")
retriever = HybridRetriever(chunks, embedder, reranker=reranker)
print("Retriever ready. base_chunks=", len(retriever.base_chunks), "aliases=", len(retriever.alias_records))
print("SUCCESS: index built.")


## 18. Demo ask

After the answer, optionally rate it:
```python
give_feedback(result['run_id'], good=True)
# or
give_feedback(result['run_id'], correction='...')
```


In [ ]:
result = ask("Can I install Mihon on Android?")
show_report(result)
display(recent_runs(10))


## Limitations

- **No weight updates** — improvement is memory, aliases, lessons, and retrieval gates only.
- **Gemma is gated** — needs HF license accept + `HF_TOKEN` secret; E4B in 8-bit fits a T4 with room for embedder/reranker/NLI.
- **NLI ≠ truth** — faithfulness is support-by-evidence, not factual correctness.
- **Crawl freshness** — corpus cache can go stale; delete `mihon_corpus.json` to refresh.
- **Aliases trust feedback** — bad thumbs-up can poison retrieval; review `give_feedback` carefully.
- **Multimodal stack** — we use text-only chat; audio/vision paths are unused.
- **Speed** — each `ask()` may call Gemma several times (route, HyDE, generate, rewrite/gap). Set `USE_HYDE=False` / `USE_ROUTE=False` to cut latency.
